### Problema 2: Análisis de Vulnerabilidades con Web Scraping

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, date
from dateutil.relativedelta import relativedelta

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE_URL = "https://lists.debian.org/debian-security-announce/2026/"
FECHA_FIN = date.today()
FECHA_INICIO = FECHA_FIN - relativedelta(months=3)

print("Fecha inicio:", FECHA_INICIO)
print("Fecha fin:", FECHA_FIN)


# ============================================================
# FUNCIÓN PARA EXTRAER LOS DATOS DE UN AVISO
# ============================================================

def extraer_datos(texto):

    # ========================================================
    # FECHA
    # ========================================================

    fecha_match = re.search(
        r"Debian Security Advisory DSA-\d+-\d+.*?\n.*?\n.*?\n([A-Z][a-z]+ \d{1,2}, \d{4})",
        texto,
        re.DOTALL
    )

    fecha = None

    if fecha_match:
        fecha = datetime.strptime(
            fecha_match.group(1),
            "%B %d, %Y"
        ).date()


    # ========================================================
    # PACKAGE
    # ========================================================

    package_match = re.search(
        r"Package\s*:\s*(.+)",
        texto
    )

    package = (
        package_match.group(1).strip()
        if package_match
        else None
    )


    # ========================================================
    # CVE
    # ========================================================

    """cve_match = re.search(
        r"CVE ID\s*:\s*(.+)",
        texto
    )

    cves = (
        cve_match.group(1).strip()
        if cve_match
        else None
    )"""
    import re

    # 1. Definimos la nueva función arriba en tu script
    def extraer_todos_los_cves(texto_bloque):
        if not texto_bloque:
            return "not yet available"
        # Busca todos los patrones CVE-AAAA-NNNN sin importar los saltos de línea
        cves_encontrados = re.findall(r"CVE-\d{4}-\d+", texto_bloque)
        # Quitamos duplicados por si acaso
        cves_unicos = list(dict.fromkeys(cves_encontrados))
        
        if cves_unicos:
            return " ".join(cves_unicos)
        return "not yet available"


    # =====================================================================
    # 2. ASÍ LO APLICÁS EN TU BUCLE DE SCRAPING:
    # =====================================================================

    # Primero buscamos dónde empieza el bloque de CVEs
    cve_match = re.search(r"CVE ID\s*:\s*([\s\S]+?)(?=\n[A-Z][a-z]+|\r?\n-{3,}|$)", texto)

    if cve_match:
        # Capturamos todo el bloque de texto (incluyendo sus saltos de línea)
        bloque_cves_sucio = cve_match.group(1)
        
        # ¡AQUÍ MANDÁS LA FUNCIÓN NUEVA!: Limpia y extrae absolutamente todos los CVEs de ese bloque
        cves = extraer_todos_los_cves(bloque_cves_sucio)
    else:
        cves = "not yet available"



    # ========================================================
    # DESCRIPCIÓN
    # ========================================================

    descripcion = None

    # Buscamos dónde termina el bloque de metadatos.
    # CVE ID es obligatorio en la mayoría de los avisos,
    # pero algunos pueden decir "not yet available".

    inicio_match = re.search(
        r"CVE ID\s*:\s*.*?\n",
        texto
    )

    if inicio_match:

        inicio = inicio_match.end()

        # Sacamos el texto posterior
        texto_posterior = texto[inicio:]

        # Eliminamos posibles campos adicionales de metadatos
        texto_posterior = re.sub(
            r"^\s*Debian Bug\s*:.*?\n",
            "",
            texto_posterior,
            flags=re.MULTILINE
        )

        # Eliminamos espacios y saltos iniciales
        texto_posterior = texto_posterior.strip()


        # ====================================================
        # BUSCAR DÓNDE TERMINA LA DESCRIPCIÓN
        # ====================================================

        patrones_fin = [
            r"\nFor the stable distribution",
            r"\nFor the oldstable distribution",
            r"\nFor the oldoldstable distribution",
            r"\nWe recommend that you upgrade",
            r"\nFor the detailed security status",
        ]

        posiciones = []

        for patron in patrones_fin:

            match_fin = re.search(
                patron,
                texto_posterior
            )

            if match_fin:
                posiciones.append(match_fin.start())


        if posiciones:

            fin = sorted(posiciones)[0]

            descripcion = texto_posterior[:fin].strip()

        else:

            descripcion = texto_posterior.strip()


        # Limpiamos saltos de línea
        descripcion = " ".join(
            descripcion.split()
        )


        return {
            "fecha": fecha,
            "package": package,
            "cves": cves,
            "descripcion": descripcion
        }

    # -------------------------
    # CVE
    # -------------------------

    cve_match = re.search(
        r"CVE ID\s*:\s*(.+)",
        texto
    )

    cves = (
        cve_match.group(1).strip()
        if cve_match
        else None
    )


    # -------------------------
    # Descripción
    # -------------------------

    descripcion_match = re.search(
        r"Debian Bug\s*:.*?\n\n(.*?)\n\nFor the stable distribution",
        texto,
        re.DOTALL
    )

    descripcion = None

    if descripcion_match:
        descripcion = " ".join(
            descripcion_match.group(1).split()
        )


    return {
        "fecha": fecha,
        "package": package,
        "cves": cves,
        "descripcion": descripcion
    }



Fecha inicio: 2026-06-17
Fecha fin: 2026-09-17

Cantidad de avisos encontrados: 416

[1/416] Procesando msg00415.html
  Fecha: 2026-09-16
  Package: thunderbird
  CVE: CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 CVE-2026-92008

[2/416] Procesando msg00414.html
  Fecha: 2026-09-16
  Package: mkvtoolnix
  CVE: CVE-2026-90783

[3/416] Procesando msg00413.html
  Fecha: 2026-09-16
  Package: firefox-esr
  CVE: CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 CVE-2026-92008

[4/416] Procesando msg00412.html
  Fecha: 2026-09-16
  Package: tor
  CVE: not yet available

[5/416] Procesando msg00411.html
  Fecha: 2026-09-16
  Package: nginx
  CVE: None

[6/416] Procesando msg00410.html
  Fecha: 2026-09-15
  Package: cjose
  CVE: CVE-2026-53938 CVE-2026-53939

[7/416] Procesando msg00409.html
  Fecha: 2026-09-14
  Package: network-manager-l2tp
  CVE: CVE-2026-19624 CVE-2026-75131 CVE-2026-75883

[8/416] Procesando msg00408.html
  Fecha: 2026-09-12
  Package: xorg-server
  CVE: CVE-2026-55999 CVE-2026

In [ ]:
# ============================================================
# OBTENER LINKS DEL ARCHIVO 2026
# ============================================================

response = requests.get(BASE_URL)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


links = []

for a in soup.find_all("a"):

    href = a.get("href")

    if href and re.match(r"msg\d+\.html$", href):
        links.append(href)


print("\nCantidad de avisos encontrados:", len(links))


# ============================================================
# ORDENAR LOS AVISOS POR NÚMERO
# ============================================================

links.sort(
    key=lambda x: int(
        re.search(r"msg(\d+)\.html", x).group(1)
    ),
    reverse=True
)

In [ ]:
# ============================================================
# RECORRER LOS AVISOS
# ============================================================

resultados = []


for i, link in enumerate(links):

    url = BASE_URL + link

    print(f"\n[{i + 1}/{len(links)}] Procesando {link}")

    try:

        response = requests.get(url)

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        texto = soup.get_text("\n", strip=True)

        datos = extraer_datos(texto)


        # Si no pudimos obtener la fecha,
        # continuamos con el siguiente aviso

        if datos["fecha"] is None:

            print("  No se pudo obtener la fecha")
            continue


        print("  Fecha:", datos["fecha"])
        print("  Package:", datos["package"])
        print("  CVE:", datos["cves"])


        # ====================================================
        # FILTRO DE LOS ÚLTIMOS 3 MESES
        # ====================================================

        if datos["fecha"] > FECHA_FIN:

            continue


        if datos["fecha"] < FECHA_INICIO:

            print("  Aviso anterior a los últimos 3 meses.")
            print("  Deteniendo scraping.")

            break


        # ====================================================
        # GUARDAR
        # ====================================================

        datos["dsa"] = re.search(
            r"DSA-\d+-\d+",
            texto
        ).group(0)

        datos["url"] = url

        resultados.append(datos)


    except requests.RequestException as e:

        print("  Error:", e)


In [ ]:
# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(resultados)


# Ordenamos por fecha

df = df.sort_values(
    "fecha",
    ascending=False
).reset_index(drop=True)


print("\n===================================")
print("RESULTADO")
print("===================================")

print("Cantidad de avisos:", len(df))

print(df.head())


# ============================================================
# GUARDAR CSV
# ============================================================

df.to_csv(
    "debian_security_advisories_ultimos_3_meses.csv",
    index=False
)

print(
    "\nArchivo guardado como:",
    "debian_security_advisories_ultimos_3_meses.csv"
)

Clasificacion en los tipos de vulnerabilidades (JUSTIFICAR)

In [ ]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. Catálogo enriquecido con mapeo semántico para mejorar la coincidencia

categorias_mapeo = {

    "Denial of Service": [
        "Denial of Service",
        "service disruption",
        "crash",
        "infinite loop",
        "resource exhaustion"
    ],

    "Remote Code Execution": [
        "Remote Code Execution",
        "execution of arbitrary code",
        "arbitrary code execution",
        "execute arbitrary code"
    ],

    "Privilege Escalation": [
        "Privilege Escalation",
        "escalate privileges",
        "gain root privileges",
        "gain elevated privileges"
    ],

    "Information Disclosure": [
        "Information Disclosure",
        "information disclosure",
        "memory disclosure",
        "sensitive information disclosure",
        "information leak",
        "disclosure of local files"
    ],

    "Authentication Bypass": [
        "Authentication Bypass",
        "authentication bypass",
        "incorrect authentication",
        "bypass authentication",
        "credential verification bypass"
    ],

    "Authorization/Access Control Bypass": [
        "Authorization Bypass",
        "Access Control Bypass",
        "authorization bypass",
        "access control bypass",
        "bypass access restrictions",
        "bypass security restrictions"
    ],

    "Sandbox Escape": [
        "Sandbox Escape",
        "sandbox escape",
        "escape the sandbox"
    ],

    "SQL Injection": [
        "SQL Injection",
        "SQL injection"
    ],

    "Cross-Site Scripting": [
        "Cross-Site Scripting",
        "cross-site scripting",
        "XSS"
    ],

    "Server-Side Request Forgery": [
        "Server-Side Request Forgery",
        "server-side request forgery",
        "SSRF"
    ],

    "Path Traversal": [
        "Path Traversal",
        "path traversal",
        "directory traversal"
    ],

    "Command Injection": [
        "Command Injection",
        "command injection",
        "execution of arbitrary commands"
    ],

    "Request Smuggling": [
        "Request Smuggling",
        "request smuggling",
        "HTTP request smuggling"
    ],

    "Header Injection": [
        "Header Injection",
        "header injection"
    ],

    "Spoofing": [
        "Spoofing",
        "spoofing",
        "UI spoofing"
    ],

    "Cryptographic/Encryption Bypass": [
        "Cryptographic Bypass",
        "Encryption Bypass",
        "encryption bypass",
        "cryptographic bypass",
        "plaintext recovery"
    ]
}

In [ ]:
# Cargar modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Pre-calcular embeddings usando la frase principal de cada categoría
nombres_categorias = list(categorias_mapeo.keys())
embeddings_categorias = model.encode(nombres_categorias)

def limpiar_y_clasificar(texto, umbral=0.28):
    if not texto or pd.isna(texto):
        return []
    
    # PASO 1: Limpieza drástica de patrones CVE y caracteres basura para eliminar ruido
    texto_limpio = re.sub(r'CVE-\d{4}-\d+', '', texto)
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()
    
    # PASO 2: Clasificación por Embeddings con umbral optimizado (0.28)
    embedding_texto = model.encode([texto_limpio])
    similitudes = cosine_similarity(embedding_texto, embeddings_categorias)[0]
    
    categorias_detectadas = set()
    for i, score in enumerate(similitudes):
        if score >= umbral:
            categorias_detectadas.add(nombres_categorias[i])
            
    # PASO 3: Respaldo por palabras clave para capturar términos explícitos omitidos
    texto_minusculas = texto.lower()
    for cat, sinonimos in categorias_mapeo.items():
        for sinonimo in sinonimos:
            if sinonimo.lower() in texto_minusculas:
                categorias_detectadas.add(cat)
                break
                
    return list(categorias_detectadas)

# =====================================================================
# Aplicar al DataFrame original
# =====================================================================
df["categorias_list"] = df["descripcion"].apply(lambda x: limpiar_y_clasificar(x))

# =====================================================================
# 5. Aplicar al DataFrame (Simulación con tus datos estructurados)
# =====================================================================

# Para facilitar la exportación a Looker Studio, creamos una cadena separada por comas
df["tipo_vulnerabilidad"] = df["categorias_list"].apply(lambda x: ", ".join(x) if x else "Unclassified")

# Mostramos el resultado estructurado
df.head(10)
# Guardamos en Resultados.csv
df.to_csv("Resultados.csv")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4022.73it/s]


,fecha,package,cves,descripcion,dsa,url,categorias_list,tipo_vulnerabilidad
0,2026-09-16,thunderbird,CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 C...,CVE-2026-92009 CVE-2026-92010 CVE-2026-92011 C...,DSA-6503-1,https://lists.debian.org/debian-security-annou...,[Remote Code Execution],Remote Code Execution
1,2026-09-16,tor,not yet available,Multiple security vulnerabilities were discove...,DSA-6500-1,https://lists.debian.org/debian-security-annou...,"[Authentication Bypass, Spoofing, Cryptographi...","Authentication Bypass, Spoofing, Cryptographic..."
2,2026-09-16,nginx,None,The update for nginx released as DSA 6496-1 ca...,DSA-6496-2,https://lists.debian.org/debian-security-annou...,[],Unclassified
3,2026-09-16,mkvtoolnix,CVE-2026-90783,A buffer overflow was found in the ODML parser...,DSA-6502-1,https://lists.debian.org/debian-security-annou...,[Remote Code Execution],Remote Code Execution
4,2026-09-16,firefox-esr,CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 C...,CVE-2026-92009 CVE-2026-92010 CVE-2026-92011 C...,DSA-6501-1,https://lists.debian.org/debian-security-annou...,"[Information Disclosure, Privilege Escalation,...","Information Disclosure, Privilege Escalation, ..."
...,...,...,...,...,...,...,...,...
151,2026-06-19,gst-libav1.0,CVE-2026-52717,It was discovered that incorrect memory manage...,DSA-6353-1,https://lists.debian.org/debian-security-annou...,[],Unclassified
152,2026-06-18,chromium,CVE-2026-12437 CVE-2026-12438 CVE-2026-12439 C...,CVE-2026-12441 CVE-2026-12442 CVE-2026-12443 C...,DSA-6351-1,https://lists.debian.org/debian-security-annou...,"[Information Disclosure, Remote Code Execution...","Information Disclosure, Remote Code Execution,..."
153,2026-06-18,thunderbird,CVE-2026-12289 CVE-2026-12290 CVE-2026-12291 C...,CVE-2026-12294 CVE-2026-12295 CVE-2026-12296 C...,DSA-6351-1,https://lists.debian.org/debian-security-annou...,[Remote Code Execution],Remote Code Execution
154,2026-06-17,firefox-esr,CVE-2026-12289 CVE-2026-12290 CVE-2026-12291 C...,CVE-2026-12294 CVE-2026-12295 CVE-2026-12296 C...,DSA-6350-1,https://lists.debian.org/debian-security-annou...,"[Privilege Escalation, Server-Side Request For...","Privilege Escalation, Server-Side Request Forg..."


In [ ]:
# 1. Aseguramos que la columna sea una lista real de Python
# 2. Aplicamos explode para duplicar filas por cada elemento de la lista
df_looker_studio = df.explode("categorias_list")

# 3. Renombramos la columna para que quede clara en el tablero de control
df_looker_studio = df_looker_studio.rename(columns={"categorias_list": "vulnerabilidad_especifica"})

# 4. Rellenamos los vacíos que hayan quedado con 'Unclassified'
df_looker_studio["vulnerabilidad_especifica"] = df_looker_studio["vulnerabilidad_especifica"].fillna("Unclassified")

# 5. Exportamos el CSV limpio listo para conectar a Google Drive / Looker Studio
df_looker_studio.to_csv("debian_vulnerabilities_looker.csv", index=False)

print(f"Dataset expandido generado. Filas originales: {len(df)} | Filas expandidas para Looker: {len(df_looker_studio)}")


Dataset expandido generado. Filas originales: 156 | Filas expandidas para Looker: 398


In [ ]:
df_looker_studio.head(10)